# Mergere genre_groups and movies data

In [85]:
import pandas as pd
import numpy as np
genre_groups_raw = pd.read_csv('data/genre_groups.csv')
genre_groups = genre_groups_raw.copy()

movies_raw = pd.read_csv('data/movies_cleaned.csv',parse_dates=['Release Date'])
movies = movies_raw.copy()

In [86]:
genre_groups.info()
genre_groups.head(3)
genre_groups['Major Genre'].unique().tolist()
genre_groups['Genre group'].unique().tolist()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Major Genre  12 non-null     str  
 1   Genre group  12 non-null     str  
dtypes: str(2)
memory usage: 324.0 bytes


['Drama',
 'Comedy & music',
 'Action & adventure',
 'Thriller & horror',
 'Non-fiction']

Before Doing merge
1. check missing values for matching rows
2. Check if Major Genre in movies has matching values in genre_groups 'Major Genre'
3. Check if 'Genre group' in movies has matching values in genre_groups 'Genre Group'

In [87]:
print(movies['Major Genre'].isna().sum())
movies['Major Genre'].isin(genre_groups['Major Genre'].tolist()).sum()

275


np.int64(2926)

Observation:
1. There are 275 rows in movies are missing 'Major Genre'
2. There 2926 movies has matching 'Major Genre' in genre_groups
3. Will join the table using leftjoin so 275 movies with missing Major Genre will be retained

In [88]:
movies_merged = pd.merge(movies,genre_groups,on='Major Genre',how='left')
movies_merged.shape

(3201, 20)

How many movies missing Genre group:
1. My prediction is is the same 275 columns where the 'Major Genre' is missing.

In [89]:
movies_merged['Genre group'].isna().sum()

np.int64(275)

What are the new columns that we may need for easy accessing data to analysis
1. Release year(For comparing missing data relationship, grouping movies by year)
2. Profit/Loss based on WorldwideGross - Production Budget(column1 - Profit(number),column2 - Profit Status(Profit/Loss))

In [90]:
movies_merged['Release Year'] = movies_merged['Release Date'].dt.year
movies_merged['Release Year'].shape
movies_merged[['Release Date','Release Year']]

,Release Date,Release Year
0,1998-06-12,1998
1,1998-08-07,1998
2,1998-08-28,1998
3,1998-09-11,1998
4,1998-10-09,1998
...,...,...
3196,2008-10-31,2008
3197,2007-03-02,2007
3198,2006-08-11,2006
3199,2005-10-28,2005


In [91]:
movies_merged['Profit'] = movies_merged['Worldwide Gross'] - movies_merged['Production Budget']
print(movies_merged['Profit'].shape)
print(movies_merged['Profit'].head(5))
movies_merged['Profit Status'] = np.where(movies_merged['Profit'] < 0.0, 'Loss','Profit')
movies_merged[movies_merged['Title'] == 'High Plains Drifter']
empty_profit_mask = movies_merged['Profit'].isna()
movies_merged.loc[empty_profit_mask,'Profit Status'] = np.nan
movies_merged[movies_merged['Title'] == 'High Plains Drifter']

(3201,)
0   -7853917.0
1    -289124.0
2     -46866.0
3      73615.0
4      87521.0
Name: Profit, dtype: float64


,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,...,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes,Title_cleaned,Missing_DVD,Missing_Running_Time_min,Genre group,Release Year,Profit,Profit Status
433,High Plains Drifter,NaN,NaN,NaN,NaN,1972-01-01,NaN,NaN,Universal,Original Screenplay,...,NaN,7.6,15718.0,high plains drifter,True,True,Action & adventure,1972,NaN,NaN


In [92]:
movies_merged.to_csv('data/movies_merged.csv',index=False)